In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1 — Setup
# ─────────────────────────────────────────────────────────────

# ── Mount Google Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Install dependencies if needed ──────────────────────────
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import sklearn
except ImportError:
    pip_install('scikit-learn')

try:
    import torchvision
except ImportError:
    pip_install('torchvision')

# ── All imports ───────────────────────────────────────────────
import os
import re
import json
import random
import copy
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms

from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_curve, auc
import pandas as pd

# ── Paths ─────────────────────────────────────────────────────
STRUCTURED_DIR = '/content/drive/MyDrive/HandVein_Project/processed/'
MODELS_DIR     = '/content/drive/MyDrive/HandVein_Project/models/'
RESULTS_DIR    = '/content/drive/MyDrive/HandVein_Project/results/'

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Variant selector ─────────────────────────────────────────
# ╔══════════════════════════════════════════════════════════╗
# ║  SET THIS BEFORE EACH RUN: 'roi' | 'blackhat' | 'frangi'║
# ╚══════════════════════════════════════════════════════════╝
VARIANT = 'roi'   # <── change to 'blackhat' or 'frangi' for subsequent runs

print(f'\n>>> VARIANT for this run: {VARIANT.upper()} <<<\n')

# ── Global constants ─────────────────────────────────────────
RANDOM_SEED    = 42
N_CLASSES      = 226
N_DB1_EXPECTED = 904
N_DB2_EXPECTED = 678
IMG_SIZE       = 128
BATCH_SIZE     = 32
EPOCHS         = 50
LR             = 1e-4
PATIENCE       = 15

# ── Seed everything ──────────────────────────────────────────
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── Device ───────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device       : {device}')
if device.type == 'cuda':
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch      : {torch.__version__}')
print(f'Torchvision  : {torchvision.__version__}')
print()
print(f'STRUCTURED_DIR : {STRUCTURED_DIR}')
print(f'MODELS_DIR     : {MODELS_DIR}')
print(f'RESULTS_DIR    : {RESULTS_DIR}')

Mounted at /content/drive

>>> VARIANT for this run: ROI <<<

Device       : cuda
GPU          : Tesla T4
VRAM         : 15.6 GB
PyTorch      : 2.11.0+cu128
Torchvision  : 0.26.0+cu128

STRUCTURED_DIR : /content/drive/MyDrive/HandVein_Project/processed/
MODELS_DIR     : /content/drive/MyDrive/HandVein_Project/models/
RESULTS_DIR    : /content/drive/MyDrive/HandVein_Project/results/


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2 — Dataset scanning and manifest building (UPDATED)
# ─────────────────────────────────────────────────────────────

# Regex: captures person_id, db_number, hand, sample_number, variant
PATTERN = re.compile(
    r'person_(\d+)_db([12])_([LR])(\d+)_([a-zA-Z]+)\.png$',
    re.IGNORECASE
)

db1_items = []   # training pool
db2_items = []   # test pool

# Walk every subdirectory of STRUCTURED_DIR
for root, dirs, files in os.walk(STRUCTURED_DIR):
    for fname in sorted(files):
        m = PATTERN.match(fname)
        if m is None:
            continue

        pid     = m.group(1)          # '001'
        db_num  = int(m.group(2))     # 1 or 2
        hand    = m.group(3)          # 'L' or 'R'
        sample  = int(m.group(4))     # 1–4 (DB1) or 1–3 (DB2)
        variant = m.group(5).lower()  # 'roi', 'blackhat', 'frangi'

        # ── CRITICAL FILTER: Skip subjects from 114 to 138 ──
        if int(pid) > 113:
            continue

        # Only keep images matching the current variant
        if variant != VARIANT:
            continue

        class_label = f'{pid}_{hand}'  # e.g. '001_L'
        full_path   = os.path.join(root, fname)

        item = {
            'path'       : full_path,
            'person'     : pid,
            'hand'       : hand,
            'db'         : db_num,
            'sample'     : sample,
            'variant'    : variant,
            'class_label': class_label,
        }

        if db_num == 1:
            db1_items.append(item)
        else:
            db2_items.append(item)

# ── Build class_to_idx: sorted unique class labels 0..225 ───
all_classes   = sorted(set(it['class_label'] for it in db1_items + db2_items))
class_to_idx  = {cls: idx for idx, cls in enumerate(all_classes)}
idx_to_class  = {idx: cls for cls, idx in class_to_idx.items()}

# ── Per-class image counts ────────────────────────────────────
from collections import Counter

db1_counts = Counter(it['class_label'] for it in db1_items)
db2_counts = Counter(it['class_label'] for it in db2_items)

db1_per_class = list(db1_counts.values())
db2_per_class = list(db2_counts.values())

# ── Verification print ────────────────────────────────────────
print('=' * 52)
print(f'  VARIANT              : {VARIANT.upper()}')
print('=' * 52)
print(f'  DB1 images found     : {len(db1_items):<6}  (expected {N_DB1_EXPECTED})')
print(f'  DB2 images found     : {len(db2_items):<6}  (expected {N_DB2_EXPECTED})')
print(f'  Classes found        : {len(all_classes):<6}  (expected {N_CLASSES})')
print(f'  DB1 images/class     : min={min(db1_per_class)}  max={max(db1_per_class)}  '
      f'mean={np.mean(db1_per_class):.2f}')
print(f'  DB2 images/class     : min={min(db2_per_class)}  max={max(db2_per_class)}  '
      f'mean={np.mean(db2_per_class):.2f}')
print('=' * 52)

# ── Assertions — raise immediately if counts are wrong ───────
assert len(db1_items) == N_DB1_EXPECTED, (
    f'DB1 count mismatch: expected {N_DB1_EXPECTED}, got {len(db1_items)}.\n'
    f'Check STRUCTURED_DIR path and VARIANT={VARIANT}.'
)
assert len(db2_items) == N_DB2_EXPECTED, (
    f'DB2 count mismatch: expected {N_DB2_EXPECTED}, got {len(db2_items)}.\n'
    f'Check STRUCTURED_DIR path and VARIANT={VARIANT}.'
)
assert len(all_classes) == N_CLASSES, (
    f'Class count mismatch: expected {N_CLASSES}, got {len(all_classes)}.'
)

print('\n  ✓ All dataset assertions passed successfully.')
print(f'  Sample DB1 items (first 3):')
for it in db1_items[:3]:
    print(f'    {os.path.basename(it["path"])}  →  class {it["class_label"]} (idx {class_to_idx[it["class_label"]]})')
print(f'  Sample DB2 items (first 3):')
for it in db2_items[:3]:
    print(f'    {os.path.basename(it["path"])}  →  class {it["class_label"]} (idx {class_to_idx[it["class_label"]]})')

  VARIANT              : ROI
  DB1 images found     : 904     (expected 904)
  DB2 images found     : 678     (expected 678)
  Classes found        : 226     (expected 226)
  DB1 images/class     : min=4  max=4  mean=4.00
  DB2 images/class     : min=3  max=3  mean=3.00

  ✓ All dataset assertions passed successfully.
  Sample DB1 items (first 3):
    person_001_db1_L1_roi.png  →  class 001_L (idx 0)
    person_001_db1_L2_roi.png  →  class 001_L (idx 0)
    person_001_db1_L3_roi.png  →  class 001_L (idx 0)
  Sample DB2 items (first 3):
    person_001_db2_L1_roi.png  →  class 001_L (idx 0)
    person_001_db2_L2_roi.png  →  class 001_L (idx 0)
    person_001_db2_L3_roi.png  →  class 001_L (idx 0)


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3 — Model definition
#
# ResNet-18 modifications:
#   1. conv1: 3-channel → 1-channel (average RGB pretrained weights)
#   2. fc: 512 → 226 classes
#   3. forward() always returns (logits, embedding) tuple
# ─────────────────────────────────────────────────────────────

class ResNet18Vein(nn.Module):
    """
    ResNet-18 adapted for single-channel (grayscale) vein images.

    Modifications vs stock ResNet-18:
      - conv1 : in_channels 3 → 1  (weights averaged from pretrained RGB)
      - fc    : out_features 1000 → n_classes

    forward() returns (logits [B, n_classes], embedding [B, 512])
    """

    def __init__(self, n_classes: int = 226):
        super().__init__()

        # ── Load pretrained ResNet-18 ────────────────────────
        base = torchvision.models.resnet18(pretrained=True)

        # ── Modification 1: conv1 RGB → grayscale ────────────
        old_weight = base.conv1.weight.data          # (64, 3, 7, 7)
        new_weight = old_weight.mean(dim=1, keepdim=True)  # (64, 1, 7, 7)

        base.conv1 = nn.Conv2d(
            in_channels=1, out_channels=64,
            kernel_size=7, stride=2, padding=3, bias=False
        )
        base.conv1.weight.data = new_weight
        print('conv1 weight transferred: averaged 3 RGB channels to 1')

        # ── Assemble feature extractor (everything before fc) ─
        self.features = nn.Sequential(
            base.conv1,
            base.bn1,
            base.relu,
            base.maxpool,
            base.layer1,
            base.layer2,
            base.layer3,
            base.layer4,
            base.avgpool,     # output: (B, 512, 1, 1)
        )

        # ── Modification 2: classification head ──────────────
        self.fc = nn.Linear(512, n_classes)

    def forward(self, x):
        """
        Parameters
        ----------
        x : Tensor  shape (B, 1, H, W)

        Returns
        -------
        logits    : Tensor  shape (B, n_classes)
        embedding : Tensor  shape (B, 512)
        """
        x         = self.features(x)           # (B, 512, 1, 1)
        embedding = torch.flatten(x, 1)        # (B, 512)
        logits    = self.fc(embedding)         # (B, n_classes)
        return logits, embedding

    def get_embedding(self, x):
        """Return 512-D embedding without gradient tracking."""
        with torch.no_grad():
            _, embedding = self.forward(x)
        return embedding


# ── Instantiate and move to device ───────────────────────────
model = ResNet18Vein(n_classes=N_CLASSES).to(device)

# ── Parameter counts (no external summary package) ───────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\nTotal parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')
print(f'Embedding dimension  : 512')
print(f'Output classes       : {N_CLASSES}')

# ── Sanity-check forward pass ─────────────────────────────────
with torch.no_grad():
    dummy      = torch.zeros(2, 1, IMG_SIZE, IMG_SIZE, device=device)
    out_logits, out_emb = model(dummy)
    print(f'\nForward pass check   : input={tuple(dummy.shape)}')
    print(f'                       logits={tuple(out_logits.shape)}')
    print(f'                       embedding={tuple(out_emb.shape)}')
assert out_logits.shape == (2, N_CLASSES), 'Logits shape mismatch'
assert out_emb.shape    == (2, 512),       'Embedding shape mismatch'
print('\n  ✓ Model instantiation and forward-pass assertions passed.')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 133MB/s]


conv1 weight transferred: averaged 3 RGB channels to 1

Total parameters     : 11,286,178
Trainable parameters : 11,286,178
Embedding dimension  : 512
Output classes       : 226

Forward pass check   : input=(2, 1, 128, 128)
                       logits=(2, 226)
                       embedding=(2, 512)

  ✓ Model instantiation and forward-pass assertions passed.


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4 — Dataset class and DataLoaders (All DB1 to Train)
# ─────────────────────────────────────────────────────────────

# ── Dataset class ────────────────────────────────────────────
class BiometricDataset(Dataset):
    """
    Parameters
    ----------
    items       : list of dicts with keys: path, class_label, ...
    class_to_idx: dict mapping class_label -> int
    transform   : torchvision transform pipeline
    """

    def __init__(self, items, class_to_idx, transform=None):
        self.items        = items
        self.class_to_idx = class_to_idx
        self.transform    = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item  = self.items[idx]
        # Load as grayscale → tensor shape (1, H, W)
        image = Image.open(item['path']).convert('L')
        if self.transform:
            image = self.transform(image)
        label = self.class_to_idx[item['class_label']]
        return image, label


# ── Transforms (CLEAN & UNDISTORTED) ─────────────────────────
# Single-channel normalisation — NEVER use ImageNet RGB stats
NORM_MEAN = [0.5]
NORM_STD  = [0.5]

# Removed Flips, Rotations, and Color Jitter to protect raw ROI structure
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

# ── Allocation: Keep ALL DB1 images for training ─────────────
train_items = db1_items  # No validation split
val_items   = []         # Keeping it empty as requested

# ── Datasets ─────────────────────────────────────────────────
train_dataset   = BiometricDataset(train_items,  class_to_idx, train_transform)
test_dataset    = BiometricDataset(db2_items,    class_to_idx, eval_transform)
gallery_dataset = BiometricDataset(db1_items,    class_to_idx, eval_transform)

# Create a safe empty validation dataset to avoid downstream NoneType errors
val_dataset     = BiometricDataset(val_items,    class_to_idx, eval_transform)

# ── DataLoaders ──────────────────────────────────────────────
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2,
    pin_memory=(device.type == 'cuda'),
    worker_init_fn=lambda wid: np.random.seed(RANDOM_SEED + wid),
    generator=g,
)

# Empty validation loader handles downstream loops cleanly without throwing errors
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2,
    pin_memory=(device.type == 'cuda'),
)

test_loader_db2 = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2,
    pin_memory=(device.type == 'cuda'),
)

gallery_loader = DataLoader(
    gallery_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2,
    pin_memory=(device.type == 'cuda'),
)

# ── Print sizes ───────────────────────────────────────────────
print('=' * 42)
print(f'  VARIANT              : {VARIANT.upper()}')
print('=' * 42)
print(f'  Training images      : {len(train_dataset)}')
print(f'  Validation images    : {len(val_dataset)}')
print(f'  Test images (DB2)    : {len(test_dataset)}')
print(f'  Gallery images (DB1) : {len(gallery_dataset)}')
print('=' * 42)
print(f'  Train batches        : {len(train_loader)}')
print(f'  Val batches          : {len(val_loader)}')
print(f'  Test batches         : {len(test_loader_db2)}')

# ── Sanity check: verify no DB2 leakage ──────────────────────
train_paths = {it['path'] for it in train_items}
test_paths  = {it['path'] for it in db2_items}
assert len(train_paths & test_paths) == 0, 'LEAKAGE: DB2 images found in training set!'
print('\n  ✓ No DB2 leakage into train set. All DB1 elements retained for training.')

# ── Visualise a few training samples ─────────────────────────
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
sample_iter = iter(train_loader)
imgs, lbls  = next(sample_iter)
for i, ax in enumerate(axes):
    img_np = imgs[i].squeeze().numpy()
    img_np = (img_np * NORM_STD[0]) + NORM_MEAN[0]   # un-normalise for display
    ax.imshow(img_np, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{idx_to_class[lbls[i].item()]}', fontsize=7)
    ax.axis('off')
fig.suptitle(f'Training samples — {VARIANT.upper()} variant', fontsize=9)
plt.tight_layout()
preview_path = os.path.join(RESULTS_DIR, f'resnet18_{VARIANT}_sample_preview.png')
plt.savefig(preview_path, dpi=100, bbox_inches='tight')
plt.close()
print(f'  Sample preview saved to: {preview_path}')

  VARIANT              : ROI
  Training images      : 904
  Validation images    : 0
  Test images (DB2)    : 678
  Gallery images (DB1) : 904
  Train batches        : 29
  Val batches          : 0
  Test batches         : 22

  ✓ No DB2 leakage into train set. All DB1 elements retained for training.
  Sample preview saved to: /content/drive/MyDrive/HandVein_Project/results/resnet18_roi_sample_preview.png


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5 — Training (Optimized for Train-Only Setup)
# ─────────────────────────────────────────────────────────────

# ── Re-seed for reproducible training ────────────────────────
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# ── Checkpoint paths ─────────────────────────────────────────
BEST_CKPT   = os.path.join(MODELS_DIR, f'resnet18_{VARIANT}_temporal_best.pth')
PERIOD_CKPT = os.path.join(MODELS_DIR, f'resnet18_{VARIANT}_temporal_epoch{{epoch}}.pth')

# ── Training objects ─────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

# ── State variables ───────────────────────────────────────────
train_losses = []
train_accs   = []
best_train_loss = float('inf')
best_epoch      = 0
start_epoch     = 1

# ── Resumable: check if best checkpoint already exists ───────
if os.path.isfile(BEST_CKPT):
    print(f'   Found existing checkpoint: {BEST_CKPT}')
    ckpt = torch.load(BEST_CKPT, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    best_train_loss = ckpt.get('best_train_loss', ckpt.get('best_val_loss', float('inf')))
    train_losses  = ckpt['train_losses']
    train_accs    = ckpt['train_accs']
    best_epoch    = ckpt['epoch']
    print(f'   Resuming from epoch {best_epoch} | best_train_loss = {best_train_loss:.4f}')
    print(f'   Skipping training — best model already available.')
    SKIP_TRAINING = True
else:
    SKIP_TRAINING = False
    print(f'   No checkpoint found. Starting fresh training.')
    print(f'   Variant: {VARIANT.upper()} | Epochs: {EPOCHS} | LR: {LR} | Batch: {BATCH_SIZE}')

# ── Training loop ─────────────────────────────────────────────
if not SKIP_TRAINING:
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        running_loss, running_correct, running_total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            logits, _ = model(images)   # Unpack tuple
            loss      = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss    += loss.item() * images.size(0)
            preds            = logits.argmax(dim=1)
            running_correct += (preds == labels).sum().item()
            running_total   += images.size(0)

        t_loss = running_loss    / running_total
        t_acc  = running_correct / running_total * 100.0

        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        train_losses.append(t_loss)
        train_accs.append(t_acc)

        # ── Save best checkpoint (Tracking lowest Training Loss) ──
        if t_loss < best_train_loss:
            best_train_loss  = t_loss
            best_epoch       = epoch
            best_model_wts   = copy.deepcopy(model.state_dict())
            torch.save({
                'epoch'               : epoch,
                'model_state_dict'    : best_model_wts,
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_train_loss'     : best_train_loss,
                'train_losses'        : train_losses,
                'train_accs'          : train_accs,
            }, BEST_CKPT)

        # ── Periodic checkpoint every 10 epochs ───────────────
        if epoch % 10 == 0:
            periodic_path = PERIOD_CKPT.format(epoch=epoch)
            torch.save({
                'epoch'               : epoch,
                'model_state_dict'    : model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_train_loss'     : best_train_loss,
                'train_losses'        : train_losses,
                'train_accs'          : train_accs,
            }, periodic_path)

        # ── Print epoch summary ───────────────────────────────
        marker = '   ★' if t_loss == best_train_loss else ''
        print(
            f'Epoch {epoch:3d}/{EPOCHS} | '
            f'Train Loss: {t_loss:.4f} | Train Acc: {t_acc:6.2f}% | '
            f'LR: {current_lr:.2e}{marker}'
        )

    print(f'\n   Best epoch       : {best_epoch}')
    print(f'   Best train loss  : {best_train_loss:.4f}')

    # ── Restore best weights ──────────────────────────────────
    model.load_state_dict(best_model_wts)
    print('   Best model weights restored.')

    # ── Plot training curves ──────────────────────────────────
    epochs_range = range(1, len(train_losses) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs_range, train_losses, label='Train Loss', color='steelblue', lw=2)
    ax1.axvline(best_epoch, color='green', ls='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'Loss Curve — {VARIANT.upper()}')
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(epochs_range, train_accs, label='Train Acc', color='steelblue', lw=2)
    ax2.axvline(best_epoch, color='green', ls='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title(f'Accuracy Curve — {VARIANT.upper()}')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.suptitle(
        f'ResNet-18 Training Curves ({VARIANT.upper()}) — Full DB1 Training',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    curves_path = os.path.join(RESULTS_DIR, f'resnet18_{VARIANT}_training_curves.png')
    plt.savefig(curves_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'   Training curves saved to: {curves_path}')

   Found existing checkpoint: /content/drive/MyDrive/HandVein_Project/models/resnet18_roi_temporal_best.pth
   Resuming from epoch 50 | best_train_loss = 0.0078
   Skipping training — best model already available.


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — Multiclass Identification Evaluation (FIXED)
# ─────────────────────────────────────────────────────────────

# ── Load best checkpoint safely ──────────────────────────────
print(f'   Loading best checkpoint: {BEST_CKPT}')
ckpt = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])

# Dynamic fallback mechanism prevents KeyError crashes
best_loss_val = ckpt.get('best_train_loss', ckpt.get('best_val_loss', 0.0))
loss_label = "best_train_loss" if 'best_train_loss' in ckpt else "best_val_loss"

print(f'   Checkpoint from epoch : {ckpt["epoch"]}')
print(f'   Saved {loss_label:<17} : {best_loss_val:.4f}')

model.eval()

# ── Run inference on DB2 ─────────────────────────────────────
all_preds, all_labels = [], []
all_probs             = []
all_top5              = []

with torch.no_grad():
    for images, labels in test_loader_db2:
        images = images.to(device)
        logits, _ = model(images)      # always unpack tuple
        probs     = torch.softmax(logits, dim=1).cpu()

        top1_preds = probs.argmax(dim=1)
        top5_preds = probs.topk(5, dim=1).indices

        all_preds.extend(top1_preds.tolist())
        all_labels.extend(labels.tolist())
        all_probs.extend(probs.tolist())
        all_top5.extend(top5_preds.tolist())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)
all_top5   = np.array(all_top5)

# ── Metrics ───────────────────────────────────────────────────
total        = len(all_labels)
correct_top1 = (all_preds == all_labels).sum()
correct_top5 = sum(
    all_labels[i] in all_top5[i] for i in range(total)
)

top1_acc  = correct_top1 / total * 100.0
top5_acc  = correct_top5 / total * 100.0

per_class_acc = []
for c in range(N_CLASSES):
    mask = all_labels == c
    if mask.sum() > 0:
        per_class_acc.append((all_preds[mask] == c).sum() / mask.sum() * 100.0)
    else:
        per_class_acc.append(0.0)
per_class_acc = np.array(per_class_acc)
mean_per_class = per_class_acc.mean()

# ── Confusion matrix ─────────────────────────────────────────
conf_mat = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
for pred, true in zip(all_preds, all_labels):
    conf_mat[true, pred] += 1

# ── 4-panel figure ────────────────────────────────────────────
fig = plt.figure(figsize=(20, 16))
fig.suptitle(
    f'ResNet-18 Identification ({VARIANT.upper()}) — DB1→DB2 Protocol',
    fontsize=14, fontweight='bold'
)

# Panel 1: Confusion matrix
ax1 = fig.add_subplot(2, 2, 1)
cmap = LinearSegmentedColormap.from_list('vein', ['#ffffff', '#1a4f8a'])
im = ax1.imshow(conf_mat, cmap=cmap, aspect='auto')
plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
ax1.set_title('Confusion Matrix (226×226)', fontsize=11)
ax1.set_xlabel('Predicted class')
ax1.set_ylabel('True class')
ax1.set_xticks([])
ax1.set_yticks([])

# Panel 2: Per-class accuracy bar chart
ax2 = fig.add_subplot(2, 2, 2)
colors = [
    'green' if v >= 75 else ('orange' if v >= 50 else 'red')
    for v in per_class_acc
]
ax2.bar(range(N_CLASSES), per_class_acc, color=colors, width=1.0)
ax2.axhline(mean_per_class, color='black', ls='--', lw=1.5,
            label=f'Mean = {mean_per_class:.1f}%')
ax2.set_xlabel('Class index')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Per-class Accuracy', fontsize=11)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
patches = [
    mpatches.Patch(color='green',  label='≥75%'),
    mpatches.Patch(color='orange', label='50–75%'),
    mpatches.Patch(color='red',    label='<50%'),
]
ax2.legend(handles=patches, loc='lower right', fontsize=8)

# Panel 3: Summary bar chart
ax3 = fig.add_subplot(2, 2, 3)
bar_vals   = [top1_acc, top5_acc, mean_per_class]
bar_labels = ['Top-1', 'Top-5', 'Mean/Class']
bar_colors = ['#2196F3', '#4CAF50', '#FF9800']
bars = ax3.bar(bar_labels, bar_vals, color=bar_colors, width=0.5)
for bar, val in zip(bars, bar_vals):
    ax3.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{val:.2f}%', ha='center', va='bottom', fontsize=12, fontweight='bold'
    )
ax3.set_ylabel('Accuracy (%)')
ax3.set_ylim(0, 110)
ax3.set_title('Identification Summary', fontsize=11)
ax3.grid(axis='y', alpha=0.3)

# Panel 4: Summary table
ax4 = fig.add_subplot(2, 2, 4)
ax4.axis('off')
table_data = [
    ['Metric',                    'Value'],
    ['Variant',                   VARIANT.upper()],
    ['Test images',               str(total)],
    ['Test classes',              str(N_CLASSES)],
    ['Top-1 accuracy',            f'{top1_acc:.2f}%'],
    ['Top-5 accuracy',            f'{top5_acc:.2f}%'],
    ['Mean per-class accuracy',   f'{mean_per_class:.2f}%'],
    ['Correct top-1',             str(correct_top1)],
    ['Correct top-5',             str(correct_top5)],
    ['Best checkpoint epoch',     str(ckpt['epoch'])],
    [f'Best loss ({loss_label})', f'{best_loss_val:.4f}'],
]
tbl = ax4.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    cellLoc='center', loc='center',
    colWidths=[0.6, 0.4]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.2, 1.8)
for key, cell in tbl.get_celld().items():
    row, col = key
    if row == 0:
        cell.set_facecolor('#2C3E50')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#ECF0F1')
ax4.set_title('Identification Metrics', fontsize=11, pad=10)

plt.tight_layout(rect=[0, 0, 1, 0.96])
id_fig_path = os.path.join(RESULTS_DIR, f'resnet18_{VARIANT}_identification.png')
plt.savefig(id_fig_path, dpi=150, bbox_inches='tight')
plt.close()

# ── Save JSON ─────────────────────────────────────────────────
id_results = {
    'variant'                : VARIANT,
    'top1_accuracy'          : round(top1_acc,      4),
    'top5_accuracy'          : round(top5_acc,      4),
    'mean_per_class_accuracy': round(mean_per_class, 4),
    'correct_top1'           : int(correct_top1),
    'correct_top5'           : int(correct_top5),
    'test_images'            : total,
    'test_classes'           : N_CLASSES,
    'best_epoch'             : ckpt['epoch'],
    'best_loss'              : round(float(best_loss_val), 4),
}
id_json_path = os.path.join(RESULTS_DIR, f'resnet18_{VARIANT}_identification.json')
with open(id_json_path, 'w') as f:
    json.dump(id_results, f, indent=2)

# ── Print summary ─────────────────────────────────────────────
print('=' * 50)
print(f'   ResNet-18 Identification ({VARIANT.upper()}) — DB1→DB2')
print('=' * 50)
print(f'   Test images  : {total}')
print(f'   Test classes : {N_CLASSES}')
print(f'   Top-1        : {top1_acc:.2f}%')
print(f'   Top-5        : {top5_acc:.2f}%')
print(f'   Mean/Class   : {mean_per_class:.2f}%')
print('=' * 50)
print(f'   Figure saved : {id_fig_path}')
print(f'   JSON saved   : {id_json_path}')

   Loading best checkpoint: /content/drive/MyDrive/HandVein_Project/models/resnet18_roi_temporal_best.pth
   Checkpoint from epoch : 50
   Saved best_train_loss   : 0.0078
   ResNet-18 Identification (ROI) — DB1→DB2
   Test images  : 678
   Test classes : 226
   Top-1        : 73.16%
   Top-5        : 87.02%
   Mean/Class   : 73.16%
   Figure saved : /content/drive/MyDrive/HandVein_Project/results/resnet18_roi_identification.png
   JSON saved   : /content/drive/MyDrive/HandVein_Project/results/resnet18_roi_identification.json


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7 — Binary Verification Evaluation (FIXED)
# ─────────────────────────────────────────────────────────────

# ── Load best checkpoint safely ──────────────────────────────
print(f'   Loading best checkpoint: {BEST_CKPT}')
ckpt = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# Dynamic fallback mechanism prevents KeyError crashes
best_loss_val = ckpt.get('best_train_loss', ckpt.get('best_val_loss', 0.0))
loss_label = "best_train_loss" if 'best_train_loss' in ckpt else "best_val_loss"

# ─────────────────────────────────────────────────────────────
# STEP 1 — Build gallery from DB1 (enrollment)
# ─────────────────────────────────────────────────────────────
print('\n   STEP 1: Extracting DB1 gallery embeddings...')
gallery_embs   = {}   # class_label -> list of tensors
gallery_labels = []

with torch.no_grad():
    for images, labels in gallery_loader:
        images = images.to(device)
        embs   = model.get_embedding(images).cpu()  # (B, 512)

        for emb, lbl in zip(embs, labels):
            cls = idx_to_class[lbl.item()]
            gallery_embs.setdefault(cls, []).append(emb)
        gallery_labels.extend(labels.tolist())

# Verify gallery counts
n_gallery_classes = len(gallery_embs)
n_gallery_total   = sum(len(v) for v in gallery_embs.values())
print(f'   Gallery classes : {n_gallery_classes}  (expected {N_CLASSES})')
print(f'   Gallery total   : {n_gallery_total}    (expected {N_DB1_EXPECTED})')
assert n_gallery_classes == N_CLASSES,      'Gallery class count mismatch'
assert n_gallery_total   == N_DB1_EXPECTED, 'Gallery image count mismatch'

# Save gallery to disk
gallery_path = os.path.join(MODELS_DIR, f'resnet18_{VARIANT}_gallery.pt')
torch.save(gallery_embs, gallery_path)
print(f'   Gallery saved to: {gallery_path}')

# ─────────────────────────────────────────────────────────────
# STEP 2 — Extract DB2 probe embeddings
# ─────────────────────────────────────────────────────────────
print('\n   STEP 2: Extracting DB2 probe embeddings...')
probe_embs = {}   # class_label -> list of tensors

with torch.no_grad():
    for images, labels in test_loader_db2:
        images = images.to(device)
        embs   = model.get_embedding(images).cpu()

        for emb, lbl in zip(embs, labels):
            cls = idx_to_class[lbl.item()]
            probe_embs.setdefault(cls, []).append(emb)

n_probe_classes = len(probe_embs)
n_probe_total   = sum(len(v) for v in probe_embs.values())
print(f'   Probe classes   : {n_probe_classes}  (expected {N_CLASSES})')
print(f'   Probe total     : {n_probe_total}    (expected {N_DB2_EXPECTED})')
assert n_probe_classes == N_CLASSES,      'Probe class count mismatch'
assert n_probe_total   == N_DB2_EXPECTED, 'Probe image count mismatch'

# ─────────────────────────────────────────────────────────────
# STEP 3 — Compute cosine similarity scores
# ─────────────────────────────────────────────────────────────
print('\n   STEP 3: Computing similarity scores...')

def cosine_similarity(a: torch.Tensor, b: torch.Tensor) -> float:
    """Cosine similarity between two 1-D tensors."""
    return float(
        torch.dot(a, b) / (torch.norm(a) * torch.norm(b) + 1e-10)
    )

sorted_classes  = sorted(gallery_embs.keys())  # 226 classes

genuine_scores  = []
impostor_scores = []

for cls in sorted_classes:
    gal_list   = gallery_embs[cls]    # 4 embeddings
    probe_list = probe_embs.get(cls, [])

    if not probe_list:
        continue

    for probe in probe_list:
        # Genuine: max sim vs same class gallery
        gen_score = max(cosine_similarity(probe, g) for g in gal_list)
        genuine_scores.append(gen_score)

        # Impostor: max sim vs each OTHER class gallery
        for other_cls in sorted_classes:
            if other_cls == cls:
                continue
            imp_score = max(cosine_similarity(probe, g) for g in gallery_embs[other_cls])
            impostor_scores.append(imp_score)

genuine_scores  = np.array(genuine_scores,  dtype=np.float32)
impostor_scores = np.array(impostor_scores, dtype=np.float32)

# ── Verify counts before proceeding ──────────────────────────
N_GENUINE  = 678
N_IMPOSTOR = 152550   # 678 × 225

print(f'   Genuine  comparisons : {len(genuine_scores)}')
print(f'   Impostor comparisons : {len(impostor_scores)}')

assert len(genuine_scores)  == N_GENUINE,  \
    f'Expected {N_GENUINE} genuine scores,  got {len(genuine_scores)}'
assert len(impostor_scores) == N_IMPOSTOR, \
    f'Expected {N_IMPOSTOR} impostor scores, got {len(impostor_scores)}'
print('   ✓ Score count assertions passed.')

# ─────────────────────────────────────────────────────────────
# STEP 4 — FAR / FRR / EER sweep
# ─────────────────────────────────────────────────────────────
print('\n   STEP 4: Computing FAR/FRR/EER...')

min_score  = min(genuine_scores.min(), impostor_scores.min())
max_score  = max(genuine_scores.max(), impostor_scores.max())
thresholds = np.linspace(min_score, max_score, 500)

FAR = np.array([(impostor_scores >= t).mean() for t in thresholds])
FRR = np.array([(genuine_scores  <  t).mean() for t in thresholds])

eer_idx  = np.argmin(np.abs(FAR - FRR))
eer_val  = float((FAR[eer_idx] + FRR[eer_idx]) / 2) * 100.0
eer_thr  = float(thresholds[eer_idx])

print(f'   EER threshold : {eer_thr:.4f}')
print(f'   EER           : {eer_val:.2f}%')

# ─────────────────────────────────────────────────────────────
# STEP 5 — ROC curve and AUC
# ─────────────────────────────────────────────────────────────
print('\n   STEP 5: Computing ROC/AUC...')

y_true  = np.array([1] * N_GENUINE + [0] * N_IMPOSTOR)
y_score = np.concatenate([genuine_scores, impostor_scores])

fpr, tpr, roc_thresholds = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)
print(f'   AUC           : {roc_auc:.4f}')

# ─────────────────────────────────────────────────────────────
# STEP 6 — 4-panel figure
# ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 11))
fig.suptitle(
    f'ResNet-18 Verification ({VARIANT.upper()}) — DB1 Gallery → DB2 Probes',
    fontsize=13, fontweight='bold'
)

# Panel 1: FAR / FRR vs threshold
ax1 = fig.add_subplot(2, 2, 1)
ax1.plot(thresholds, FAR * 100, label='FAR', color='tomato',    lw=2)
ax1.plot(thresholds, FRR * 100, label='FRR', color='steelblue', lw=2)
ax1.axhline(eer_val, color='green', ls='--', lw=1.5, label=f'EER = {eer_val:.2f}%')
ax1.axvline(eer_thr, color='gray',  ls=':',  lw=1.2)
ax1.set_xlabel('Threshold')
ax1.set_ylabel('Error rate (%)')
ax1.set_title('FAR and FRR vs Threshold', fontsize=10)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Panel 2: ROC curve
ax2 = fig.add_subplot(2, 2, 2)
ax2.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {roc_auc:.4f}')
ax2.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve', fontsize=10)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1.02])

# Panel 3: Score distributions
ax3 = fig.add_subplot(2, 2, 3)
bins = np.linspace(
    min(genuine_scores.min(), impostor_scores.min()),
    max(genuine_scores.max(), impostor_scores.max()),
    60
)
ax3.hist(impostor_scores, bins=bins, color='tomato',    alpha=0.6,
         label=f'Impostor (n={N_IMPOSTOR:,})', density=True)
ax3.hist(genuine_scores,  bins=bins, color='steelblue', alpha=0.7,
         label=f'Genuine  (n={N_GENUINE:,})',  density=True)
ax3.axvline(eer_thr, color='green', ls='--', lw=1.5, label=f'EER thr = {eer_thr:.3f}')
ax3.set_xlabel('Cosine similarity score')
ax3.set_ylabel('Density')
ax3.set_title('Score Distributions', fontsize=10)
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

# Panel 4: Summary table
ax4 = fig.add_subplot(2, 2, 4)
ax4.axis('off')
table_data = [
    ['Metric',                'Value'],
    ['Variant',               VARIANT.upper()],
    ['Genuine comparisons',   f'{N_GENUINE:,}'],
    ['Impostor comparisons',  f'{N_IMPOSTOR:,}'],
    ['EER',                   f'{eer_val:.2f}%'],
    ['AUC',                   f'{roc_auc:.4f}'],
    ['EER threshold',         f'{eer_thr:.4f}'],
    ['Best epoch',            str(ckpt['epoch'])],
    [f'Best loss ({loss_label})', f'{best_loss_val:.4f}'],
]
tbl = ax4.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    cellLoc='center', loc='center',
    colWidths=[0.6, 0.4]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.2, 1.9)
for key, cell in tbl.get_celld().items():
    row, col = key
    if row == 0:
        cell.set_facecolor('#2C3E50')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#ECF0F1')
ax4.set_title('Verification Metrics', fontsize=10, pad=10)

plt.tight_layout(rect=[0, 0, 1, 0.96])
ver_fig_path = os.path.join(RESULTS_DIR, f'resnet18_{VARIANT}_verification.png')
plt.savefig(ver_fig_path, dpi=150, bbox_inches='tight')
plt.close()

# ── Save JSON ─────────────────────────────────────────────────
ver_results = {
    'variant'    : VARIANT,
    'eer'        : round(eer_val,  4),
    'auc'        : round(roc_auc,  4),
    'eer_threshold': round(eer_thr, 6),
    'n_genuine'  : N_GENUINE,
    'n_impostor' : N_IMPOSTOR,
}
ver_json_path = os.path.join(RESULTS_DIR, f'resnet18_{VARIANT}_verification.json')
with open(ver_json_path, 'w') as f:
    json.dump(ver_results, f, indent=2)

# ── Print summary ─────────────────────────────────────────────
print('=' * 50)
print(f'   ResNet-18 Verification ({VARIANT.upper()}) — DB1→DB2')
print('=' * 50)
print(f'   Genuine  comparisons : {N_GENUINE:,}')
print(f'   Impostor comparisons : {N_IMPOSTOR:,}')
print(f'   EER                  : {eer_val:.2f}%')
print(f'   AUC                  : {roc_auc:.4f}')
print('=' * 50)
print(f'   Figure saved : {ver_fig_path}')
print(f'   JSON saved   : {ver_json_path}')

   Loading best checkpoint: /content/drive/MyDrive/HandVein_Project/models/resnet18_roi_temporal_best.pth

   STEP 1: Extracting DB1 gallery embeddings...
   Gallery classes : 226  (expected 226)
   Gallery total   : 904    (expected 904)
   Gallery saved to: /content/drive/MyDrive/HandVein_Project/models/resnet18_roi_gallery.pt

   STEP 2: Extracting DB2 probe embeddings...
   Probe classes   : 226  (expected 226)
   Probe total     : 678    (expected 678)

   STEP 3: Computing similarity scores...
   Genuine  comparisons : 678
   Impostor comparisons : 152550
   ✓ Score count assertions passed.

   STEP 4: Computing FAR/FRR/EER...
   EER threshold : 0.5144
   EER           : 5.45%

   STEP 5: Computing ROC/AUC...
   AUC           : 0.9878
   ResNet-18 Verification (ROI) — DB1→DB2
   Genuine  comparisons : 678
   Impostor comparisons : 152,550
   EER                  : 5.45%
   AUC                  : 0.9878
   Figure saved : /content/drive/MyDrive/HandVein_Project/results/resnet18_roi

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 8 — Final Comparison Table (FIXED)
# ─────────────────────────────────────────────────────────────
import os
import json
import numpy as np
import pandas as pd              # Fixed: Added missing import for CSV export
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches  # Fixed: Added missing patch import for legend shapes

# ── Hardcoded baseline results ────────────────────────────────
# All trained with DB1, tested on DB2, 226 classes, same temporal protocol
baselines = {
    'SVM-RBF'      : {'rank1': 80.09, 'rank5': 84.96, 'eer': 15.56, 'auc': 0.8593},
    'LDA'          : {'rank1': 80.38, 'rank5': 88.79, 'eer': 35.56, 'auc': 0.6889},
    'Random Forest': {'rank1': 65.04, 'rank5': 83.19, 'eer': 17.78, 'auc': 0.8111},
    'VeinCNN'      : {'rank1': 51.18, 'rank5': 71.83, 'eer':  9.99, 'auc': 0.9650},
}

# ── Load ResNet-18 results from saved JSONs ───────────────────
rn18_results = {}
VARIANTS     = ['roi', 'blackhat', 'frangi']

for v in VARIANTS:
    id_path  = os.path.join(RESULTS_DIR, f'resnet18_{v}_identification.json')
    ver_path = os.path.join(RESULTS_DIR, f'resnet18_{v}_verification.json')

    if os.path.isfile(id_path) and os.path.isfile(ver_path):
        with open(id_path)  as f: id_data  = json.load(f)
        with open(ver_path) as f: ver_data = json.load(f)
        rn18_results[v] = {
            'rank1': id_data['top1_accuracy'],
            'rank5': id_data['top5_accuracy'],
            'eer'  : ver_data['eer'],
            'auc'  : ver_data['auc'],
        }
        print(f'   Loaded {v:8s}: Rank-1={rn18_results[v]["rank1"]:.2f}% | '
              f'Rank-5={rn18_results[v]["rank5"]:.2f}% | '
              f'EER={rn18_results[v]["eer"]:.2f}% | '
              f'AUC={rn18_results[v]["auc"]:.4f}')
    else:
        rn18_results[v] = None
        print(f'   WARNING: Results for variant "{v}" not found — run Cells 5–7 for this variant first.')

# ── Build rows for the comparison table ──────────────────────
rows = []

# Baselines
for model_name, res in baselines.items():
    rows.append([
        model_name,
        f'{res["rank1"]:.2f}%',
        f'{res["rank5"]:.2f}%',
        f'{res["eer"]:.2f}%',
        f'{res["auc"]:.4f}',
    ])

# ResNet-18 variants
variant_labels = {
    'roi'     : 'ResNet-18 (ROI)',
    'blackhat': 'ResNet-18 (Blackhat)',
    'frangi'  : 'ResNet-18 (Frangi)',
}
for v in VARIANTS:
    label = variant_labels[v]
    if rn18_results[v] is not None:
        res = rn18_results[v]
        rows.append([
            label,
            f'{res["rank1"]:.2f}%',
            f'{res["rank5"]:.2f}%',
            f'{res["eer"]:.2f}%',
            f'{res["auc"]:.4f}',
        ])
    else:
        rows.append([label, 'N/A', 'N/A', 'N/A', 'N/A'])

col_headers = ['Model', 'Rank-1 ↑', 'Rank-5 ↑', 'EER ↓', 'AUC ↑']
n_rows = len(rows)
n_cols = len(col_headers)

# ── Extract numeric values for highlighting ───────────────────
def extract_float(s):
    """Extract float from '80.09%' or '0.8593'; return None if N/A."""
    if s in ('N/A', ''):
        return None
    return float(s.replace('%', ''))

# column index -> (list of values per row, higher_is_better)
col_specs = [
    (1, True),   # Rank-1
    (2, True),   # Rank-5
    (3, False),  # EER  (lower is better)
    (4, True),   # AUC
]

best_cell  = {}   # (row, col) -> green
worst_cell = {}   # (row, col) -> red

for col_idx, higher_better in col_specs:
    vals = [extract_float(rows[r][col_idx]) for r in range(n_rows)]
    valid_vals = [(v, i) for i, v in enumerate(vals) if v is not None]
    if not valid_vals:
        continue
    best_val  = max(v for v, _ in valid_vals) if higher_better else min(v for v, _ in valid_vals)
    worst_val = min(v for v, _ in valid_vals) if higher_better else max(v for v, _ in valid_vals)
    for v, r in valid_vals:
        if v == best_val:
            best_cell[(r, col_idx)]  = True
        if v == worst_val:
            worst_cell[(r, col_idx)] = True

# ── Build figure ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 0.55 * (n_rows + 2) + 2.5))
ax.axis('off')

tbl = ax.table(
    cellText=rows,
    colLabels=col_headers,
    cellLoc='center', loc='center',
    colWidths=[0.28, 0.17, 0.17, 0.17, 0.17],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.3, 2.4)

# Style: header
for col in range(n_cols):
    cell = tbl[(0, col)]
    cell.set_facecolor('#2C3E50')
    cell.set_text_props(color='white', fontweight='bold')

# Style: separator between baselines and ResNet-18 variants
N_BASELINE = len(baselines)

for row in range(1, n_rows + 1):  # matplotlib table rows are 1-indexed (0=header)
    for col in range(n_cols):
        cell = tbl[(row, col)]
        r0   = row - 1  # 0-indexed row

        # Default alternating row colour
        if r0 < N_BASELINE:
            base_color = '#F8F9FA' if r0 % 2 == 0 else '#FFFFFF'
        else:
            base_color = '#EAF4FF' if r0 % 2 == 0 else '#D6ECFF'

        cell.set_facecolor(base_color)

        # Best (green) / worst (red) highlights for metric columns
        if col > 0:
            if (r0, col) in best_cell:
                cell.set_facecolor('#C8F7C5')   # light green
                cell.set_text_props(fontweight='bold')
            elif (r0, col) in worst_cell:
                cell.set_facecolor('#FADBD8')   # light red

        # Bold model name for ResNet-18 variants
        if col == 0 and r0 >= N_BASELINE:
            cell.set_text_props(fontweight='bold')

fig.suptitle(
    'Hand Vein Recognition — Model Comparison\n'
    'Protocol: DB1 Train → DB2 Test | 226 Classes | Cross-Session Temporal Split',
    fontsize=12, fontweight='bold', y=0.98
)

# Legend
legend_patches = [
    mpatches.Patch(color='#C8F7C5', label='Best in column'),
    mpatches.Patch(color='#FADBD8', label='Worst in column'),
    mpatches.Patch(color='#D6ECFF', label='ResNet-18 variants'),
]
ax.legend(
    handles=legend_patches,
    loc='lower right', fontsize=9,
    framealpha=0.9
)

plt.tight_layout()
cmp_fig_path = os.path.join(RESULTS_DIR, 'resnet18_comparison_table.png')
plt.savefig(cmp_fig_path, dpi=150, bbox_inches='tight')
plt.close()

# ── Save CSV ──────────────────────────────────────────────────
csv_rows = []
for row in rows:
    csv_rows.append({
        'Model' : row[0],
        'Rank-1': row[1],
        'Rank-5': row[2],
        'EER'   : row[3],
        'AUC'   : row[4],
    })
df = pd.DataFrame(csv_rows)
cmp_csv_path = os.path.join(RESULTS_DIR, 'resnet18_comparison_table.csv')
df.to_csv(cmp_csv_path, index=False)

# ── Print table to console ────────────────────────────────────
print('\n' + '=' * 80)
print('   Hand Vein Recognition — Final Model Comparison (DB1→DB2)')
print('=' * 80)
header = f'   {"Model":<22} {"Rank-1":>10} {"Rank-5":>10} {"EER":>10} {"AUC":>10}'
print(header)
print('   ' + '-' * 64)
for i, row in enumerate(rows):
    if i == N_BASELINE:
        print('   ' + '─' * 64 + '  ← ResNet-18 variants below')
    print(f'   {row[0]:<22} {row[1]:>10} {row[2]:>10} {row[3]:>10} {row[4]:>10}')
print('=' * 80)
print(f'\n   Figure saved : {cmp_fig_path}')
print(f'   CSV saved    : {cmp_csv_path}')

   Loaded roi     : Rank-1=73.16% | Rank-5=87.02% | EER=5.45% | AUC=0.9878

   Hand Vein Recognition — Final Model Comparison (DB1→DB2)
   Model                      Rank-1     Rank-5        EER        AUC
   ----------------------------------------------------------------
   SVM-RBF                    80.09%     84.96%     15.56%     0.8593
   LDA                        80.38%     88.79%     35.56%     0.6889
   Random Forest              65.04%     83.19%     17.78%     0.8111
   VeinCNN                    51.18%     71.83%      9.99%     0.9650
   ────────────────────────────────────────────────────────────────  ← ResNet-18 variants below
   ResNet-18 (ROI)            73.16%     87.02%      5.45%     0.9878
   ResNet-18 (Blackhat)          N/A        N/A        N/A        N/A
   ResNet-18 (Frangi)            N/A        N/A        N/A        N/A

   Figure saved : /content/drive/MyDrive/HandVein_Project/results/resnet18_comparison_table.png
   CSV saved    : /content/drive/MyDrive/Ha